# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
spaceship_cleaned = spaceship.dropna()
spaceship_cleaned.isna().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

In [4]:
spaceship_cleaned.shape

(6606, 14)

In [5]:
# Extract deck letter from Cabin
spaceship_cleaned.loc[:, "Cabin"] = spaceship_cleaned["Cabin"].str[0]

In [6]:
# drop passengerid and Name
spaceship_cleaned = spaceship_cleaned.drop(columns=["PassengerId", "Name"])

In [7]:
# do dummies for non-numerical columns
spaceship_cleaned = pd.get_dummies(spaceship_cleaned, drop_first=True)

**Perform Train Test Split**

In [8]:
from sklearn.model_selection import train_test_split

X = spaceship_cleaned.drop(columns="Transported")
y = spaceship_cleaned["Transported"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

Bagging and pasting are ensemble methods that reduce variance by training multiple models on different subsets of the data and aggregating their predictions.
- Bagging samples with replacement.
- Pasting samples without replacement.

In [11]:
#bagging
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier

bagging_clf = BaggingClassifier(
    estimator=KNeighborsClassifier(),
    n_estimators=100,
    bootstrap=True,   # False would be pasting
    random_state=42
)

bagging_clf.fit(X_train, y_train)

accuracy = bagging_clf.score(X_test, y_test)
f"{accuracy * 100:.2f}%"


'79.73%'

In [12]:
# pasting
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier

pasting_clf = BaggingClassifier(
    estimator=KNeighborsClassifier(),
    n_estimators=100,
    bootstrap=False,  # pasting
    random_state=42
)

pasting_clf.fit(X_train, y_train)

accuracy = pasting_clf.score(X_test, y_test)
f"{accuracy * 100:.2f}%"


'78.74%'

Pasting reduces variance less aggressively than bagging. It is useful when the dataset is large and bootstrap resampling adds unnecessary noise.

- Random Forests

Random Forest combines bagging with feature subsampling, reducing correlation between trees.

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_train, y_train)

accuracy = rf_clf.score(X_test, y_test)
f"{accuracy * 100:.2f}%"


'80.48%'

- Gradient Boosting

Gradient boosting reduces bias by sequentially correcting errors. It is sensitive to hyperparameters.

In [14]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_train, y_train)

accuracy = gb_clf.score(X_test, y_test)
f"{accuracy * 100:.2f}%"


'80.71%'

- Adaptive Boosting

Adaptive Boosting focuses on hard-to-classify samples.

In [15]:
from sklearn.ensemble import AdaBoostClassifier

ada_clf = AdaBoostClassifier(random_state=42)
ada_clf.fit(X_train, y_train)

accuracy = ada_clf.score(X_test, y_test)
f"{accuracy * 100:.2f}%"


'77.69%'

Which model is the best and why?

Gradient Boosting is the model performnig the best from the 4 Ensemble models tested, as it reduces bias and captures nonlinear feature interactions. 